# 📈 Phase 2: Exploratory Data Analysis (EDA) & Business Diagnostics
**Project:** E-Commerce Sales Analytics Portfolio Project
**Objective:** Uncover core revenue drivers, profitability bottlenecks, temporal patterns, regional performance disparities, and pricing elasticity using Pandas, Matplotlib, and Seaborn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 150

CLEANED_DATA_PATH = os.path.join('..', 'data', 'cleaned', 'superstore_cleaned.csv')
df = pd.read_csv(CLEANED_DATA_PATH)
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])
print(f'Cleaned dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

## 1. Executive Key Performance Indicators (KPIs)
Computing baseline high-level metrics for the executive dashboard.

In [ ]:
total_sales = df['sales'].sum()
total_profit = df['profit'].sum()
total_orders = df['order_id'].nunique()
total_qty = df['quantity'].sum()
total_customers = df['customer_id'].nunique()
aov = total_sales / total_orders
profit_margin = (total_profit / total_sales) * 100

kpis = pd.DataFrame({
    'Metric': ['Total Revenue', 'Total Profit', 'Profit Margin', 'Total Orders', 'Total Quantity Sold', 'Unique Customers', 'Average Order Value (AOV)'],
    'Value': [f'${total_sales:,.2f}', f'${total_profit:,.2f}', f'{profit_margin:.2f}%', f'{total_orders:,}', f'{total_qty:,}', f'{total_customers:,}', f'${aov:.2f}']
})
kpis

## 2. Temporal & Seasonality Analysis
Analyzing monthly revenue and profit trends over time to identify recurring peaks and seasonal slowdowns.

In [ ]:
monthly = df.groupby('order_year_month').agg({'sales': 'sum', 'profit': 'sum'}).reset_index()

plt.figure(figsize=(14, 5))
plt.plot(monthly['order_year_month'], monthly['sales'], marker='o', color='#1f77b4', label='Revenue ($)', linewidth=2)
plt.plot(monthly['order_year_month'], monthly['profit'], marker='s', color='#2ca02c', label='Profit ($)', linewidth=2)
plt.title('Monthly Revenue & Profit Growth Trajectory (2014 - 2017)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Year-Month')
plt.ylabel('Amount ($)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Category & Sub-Category Profitability Diagnostics
Breaking down performance to identify high-margin product lines versus loss leaders.

In [ ]:
subcat = df.groupby(['category', 'sub_category']).agg({
    'sales': 'sum',
    'profit': 'sum',
    'quantity': 'sum',
    'order_id': 'nunique'
}).reset_index()
subcat['profit_margin_pct'] = (subcat['profit'] / subcat['sales']) * 100
subcat = subcat.sort_values('profit', ascending=False)
subcat.style.background_gradient(subset=['profit', 'profit_margin_pct'], cmap='RdYlGn')

## 4. Discount Sensitivity & Margin Erosion
Evaluating the impact of discounting on net profitability.

In [ ]:
discount_summary = df.groupby('discount_bracket').agg({
    'sales': 'sum',
    'profit': 'sum',
    'quantity': 'sum',
    'row_id': 'count'
}).rename(columns={'row_id': 'transaction_count'}).reset_index()
discount_summary['margin_pct'] = (discount_summary['profit'] / discount_summary['sales']) * 100
discount_summary['pct_of_total_orders'] = (discount_summary['transaction_count'] / len(df)) * 100
discount_summary

## 5. Regional & Geographic Distribution
Identifying top-performing regions and states operating at a deficit.

In [ ]:
state_perf = df.groupby('state').agg({
    'sales': 'sum',
    'profit': 'sum',
    'order_id': 'nunique'
}).reset_index()
state_perf['margin_pct'] = (state_perf['profit'] / state_perf['sales']) * 100

top_states = state_perf.sort_values('profit', ascending=False).head(5)
bottom_states = state_perf.sort_values('profit', ascending=True).head(5)
print('=== TOP 5 MOST PROFITABLE STATES ===')
print(top_states[['state', 'sales', 'profit', 'margin_pct']].to_string(index=False))

print('\n=== BOTTOM 5 LOSS-MAKING STATES ===')
print(bottom_states[['state', 'sales', 'profit', 'margin_pct']].to_string(index=False))